# H&M Customer Value Analysis

Open this notebook and select **Run All**. It discovers Kaggle competition input, `data/raw/h-and-m`, or `HM_RAW_DATA_DIR` automatically. Every metric uses the full available dataset; no customer, transaction, product, or image analysis sampling is used.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    if os.environ.get("PROJECT_ROOT"):
        candidates.append(Path(os.environ["PROJECT_ROOT"]))
    candidates.append(Path("/kaggle/working/customer-value-segmentation-pipeline"))
    for candidate in candidates:
        if (candidate / "src" / "pipeline.py").is_file():
            return candidate.resolve()
    raise RuntimeError("Project source was not found. Attach or copy the repository, then set PROJECT_ROOT if needed.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import DataAnalyzer
from src.reporting import build_business_insights, summarize_numeric, summarize_rfm_segments
from src.runtime import discover_runtime

context = discover_runtime(ROOT)
analyzer = DataAnalyzer(context)
started = time.monotonic()
print(f"Runtime: {context.runtime_name}")
print("Project source: available")
print("H&M source validation: PASS")
print(f"Runtime artifact root: {context.runtime_root}")
print("Analysis scope: FULL DATASET")
print("Customer sampling: NONE")
print("Product sampling: NONE")
print("Image analysis sampling: NONE")

## Full-data preparation and multimodal features

Transactions are read in Pandas chunks. `transactions`, `customers`, `articles`, and `image_features` remain at their natural grains. This avoids repeating customer and product attributes across tens of millions of transactions; only columns required by a statistic are joined, in chunks where necessary.

Customer age is imputed once at customer grain. The median represents a typical age while being less sensitive than the mean to a skewed distribution, and `club_member_status` provides a reproducible customer grouping. The global median is a fallback when a group has no known ages. This preserves customers but can reduce variance and exaggerate apparent group differences.

Each image file is read with `matplotlib.image.imread`. The file loop performs I/O only; the image-internal calculation is NumPy vectorized over the complete decoded array with `np.mean` and `np.std`. Images are never stacked into one global tensor.

In [ ]:
summary = analyzer.load_data()
image_features = analyzer.engineer_features()
iqr = analyzer.detect_outliers()
rfm = analyzer.calculate_rfm()

transaction_schema = pd.read_csv(analyzer.transactions_path, nrows=0)
customers = pd.read_csv(analyzer.customers_path, dtype={"customer_id": "string"})
articles = pd.read_csv(analyzer.articles_path, dtype={"product_id": "string"})
inventory = pd.DataFrame([
    ["transactions", "one transaction", f"{summary['transaction_rows']:,} × {len(transaction_schema.columns)}", "date, numeric, IDs", "RFM and time series"],
    ["customers", "one customer", f"{len(customers):,} × {len(customers.columns)}", "numeric, categories, ID", "age and membership"],
    ["articles", "one product", f"{len(articles):,} × {len(articles.columns)}", "text, categories, ID", "text/category features"],
    ["image_features", "one product image", f"{len(image_features):,} × {len(image_features.columns)}", "numeric, status, ID", "full-image Mean/Std"],
], columns=["Source", "Grain", "Shape", "Primary types", "Role"])
display(Markdown("## Dataset Inventory"))
display(inventory)

transaction_preview = pd.read_csv(analyzer.transactions_path, nrows=5).copy()
transaction_preview["customer_id"] = "<masked>"
transaction_preview["product_id"] = "<masked>"
display(transaction_preview.head())
transaction_preview.info()
display(transaction_preview.describe())
print("Tables remain normalized by transaction, customer, product, and image-product grain.")
print("Image array processing: matplotlib imread + full-array NumPy np.mean/np.std")

## IQR, correlation, and visual analysis

IQR uses the full price distribution. It is robust to extreme values because it uses the middle 50%, but a naturally right-skewed fashion-price distribution can still label legitimate premium products as outliers. The before/after boxplot below is therefore a diagnostic, not an instruction to delete data.

Correlation A is full transaction/customer scope: price versus customer age. Correlation B is full product/image scope: image mean versus product-name length. A correlation describes linear association, not causation.

In [ ]:
price_values = np.memmap(
    context.aggregate_root / "unit_price_values.dat",
    dtype="float64",
    mode="r",
    shape=(summary["transaction_rows"],),
)
inlier_prices = price_values[(price_values >= iqr["lower_fence"]) & (price_values <= iqr["upper_fence"])]
price_statistics = summarize_numeric(price_values)
display(pd.DataFrame([price_statistics], index=["unit_price (full transactions)"]).round(6))
distribution_note = "평균이 중앙값보다 높아 오른쪽 꼬리 가능성을 보여준다" if price_statistics["mean"] > price_statistics["median"] else "평균과 중앙값의 관계상 강한 오른쪽 꼬리 근거는 제한적이다"
display(Markdown(
    f"**전체 거래 가격 기술통계:** 평균은 `{price_statistics['mean']:.6f}`, 중앙값은 `{price_statistics['median']:.6f}`, "
    f"표준편차는 `{price_statistics['std']:.6f}`이며 Q1–Q3는 `{price_statistics['q1']:.6f}–{price_statistics['q3']:.6f}`이다. "
    f"{distribution_note}. 따라서 평균만 보지 않고 중앙값과 사분위 범위를 함께 사용해야 한다."
))

customer_age = pd.read_csv(context.processed_root / "customers.csv", dtype={"customer_id": "string"})[["customer_id", "age"]]
pair_count = pair_x = pair_y = pair_xy = pair_x2 = pair_y2 = 0.0
monthly_parts = []
for transaction_chunk in pd.read_csv(context.processed_root / "transactions.csv", parse_dates=["order_date"], dtype={"customer_id": "string"}, chunksize=analyzer.chunksize):
    price_age_chunk = transaction_chunk[["customer_id", "unit_price"]].merge(customer_age, on="customer_id", how="left").dropna()
    x = price_age_chunk["unit_price"].to_numpy(dtype=float)
    y = price_age_chunk["age"].to_numpy(dtype=float)
    pair_count += len(x); pair_x += x.sum(); pair_y += y.sum(); pair_xy += (x * y).sum(); pair_x2 += (x * x).sum(); pair_y2 += (y * y).sum()
    monthly_parts.append(transaction_chunk.groupby(transaction_chunk["order_date"].dt.to_period("M"))["unit_price"].sum())
monthly_value = pd.concat(monthly_parts, axis=1).fillna(0).sum(axis=1)
product_features = pd.read_csv(context.feature_root / "product_images.csv", dtype={"product_id": "string"}).merge(
    pd.read_csv(context.processed_root / "articles.csv", dtype={"product_id": "string"})[["product_id", "product_name_length"]], on="product_id", how="left"
)
price_age_corr = (pair_count * pair_xy - pair_x * pair_y) / np.sqrt((pair_count * pair_x2 - pair_x ** 2) * (pair_count * pair_y2 - pair_y ** 2))
image_text_corr = product_features[["image_mean", "product_name_length"]].corr().loc["image_mean", "product_name_length"]
display(Markdown(f"**Transaction/customer scope:** price–age correlation is `r = {price_age_corr:+.3f}`. This is a linear-association measure; age alone is unlikely to explain price when the magnitude is close to zero."))
display(Markdown(f"**Product/image scope:** image-mean–name-length correlation is `r = {image_text_corr:+.3f}`. This measures association between simple visual brightness and text length, not product quality or demand."))

plt.figure(figsize=(8, 4))
plt.hist(price_values, bins=40)
plt.title("Relative Price Distribution")
plt.xlabel("Relative dataset price value")
plt.ylabel("Transaction count")
plt.show()

plt.figure(figsize=(8, 4))
plt.boxplot([price_values, inlier_prices], tick_labels=["Before IQR", "After IQR"])
plt.title("Relative Price Before vs After IQR Filtering")
plt.xlabel("IQR treatment state")
plt.ylabel("Relative dataset price value")
plt.show()

plt.figure(figsize=(8, 4))
rfm["segment"].value_counts().sort_index().plot.bar()
plt.title("RFM Segment Customer Counts")
plt.xlabel("RFM segment")
plt.ylabel("Customer count")
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(product_features[["image_mean", "image_std", "product_name_length"]].corr(), annot=True, cmap="Blues")
plt.title("Product Image and Text Feature Correlation")
plt.xlabel("Feature")
plt.ylabel("Feature")
plt.show()

plt.figure(figsize=(8, 4))
plt.scatter(rfm["frequency"], rfm["monetary"], alpha=0.15, s=3, rasterized=True)
plt.title("Customer Purchase Frequency vs Monetary Value")
plt.xlabel("Unique purchase dates")
plt.ylabel("Aggregate relative dataset value")
plt.show()

plt.figure(figsize=(9, 4))
monthly_value.sort_index().plot()
plt.title("Monthly Aggregate Relative Dataset Value")
plt.xlabel("Order month")
plt.ylabel("Aggregate relative dataset value")
plt.show()

## RFM segmentation

The analysis date is a parameter. Its default is the final transaction date plus one day, so the newest purchaser has a positive recency of one day. Frequency is the number of unique purchase dates, preventing multiple product rows bought on the same day from inflating purchase frequency. Monetary is the sum of the relative transaction value.

R, F, and M are each divided into four rank-based quantiles. Quartiles avoid inventing currency-specific business thresholds and create comparable 1–4 scores that transfer to another dataset. Lower Recency receives a higher score; higher Frequency and Monetary receive higher scores. `rank(method="first")` makes quantile assignment deterministic when customers share the same value, although customers next to a boundary should not be treated as fundamentally different.

Rules are applied in priority order: strong scores on all three dimensions become **VIP**; high-frequency customers become **Loyal**; very recent low-frequency customers become **New**; low-recency-score customers become **Churned**; the remainder become **Potential**. Business validity is assessed below from observed customer share, Monetary share, and mean R/F/M rather than from labels alone.

In [ ]:
segment_summary = summarize_rfm_segments(rfm)
display(segment_summary.round(4))
business_insights = build_business_insights(segment_summary)
display(Markdown("## Runtime Business Insights"))
display(Markdown(business_insights))
insight_path = context.artifact_root / "business_insights.md"
insight_path.write_text(business_insights, encoding="utf-8")
print("Copy-ready business insights:", insight_path)
display(Markdown("### Final execution summary"))
print("Processed transactions:", summary["transaction_rows"])
print("Processed customers:", summary["customer_rows"])
print("Processed products:", summary["product_rows"])
print("IQR outlier count:", iqr["outlier_count"])
print("Total execution time (seconds):", round(time.monotonic() - started, 1))